In [ ]:
# Loading dataset

In [20]:
%run data_loader_panel.py

Loaded: Active_companies.xlsx

  Active_companies.xlsx
  Rows: 6,400    Columns: 157

|   # | Column                                     | Dtype   |   Non-Null |   Null |
|-----|--------------------------------------------|---------|------------|--------|
|   0 | Company name Latin alphabet                | str     |      6,400 |      0 |
|   1 | Inactive                                   | str     |      6,400 |      0 |
|   2 | Quoted                                     | str     |      6,400 |      0 |
|   3 | Branch                                     | str     |      6,400 |      0 |
|   4 | OwnData                                    | str     |      6,400 |      0 |
|   5 | Woco                                       | str     |      6,400 |      0 |
|   6 | Country ISO code                           | str     |      6,400 |      0 |
|   7 | NACE Rev. 2, core code (4 digits)          | int64   |      6,400 |      0 |
|   8 | Consolidation code                         | str     |  

In [10]:
# Get unique companies and their reporting years
companies_by_year = df_panel_data.groupby('Fiscal Year')['Company Name'].unique()

# Find companies present in each year, dropping NaN values
companies_by_year_dict = {year: set(c for c in companies if pd.notna(c)) for year, companies in companies_by_year.items()}

# Sort years to compare consecutive years
sorted_years = sorted(companies_by_year_dict.keys())

# Find companies that stopped reporting
disappeared_companies = {}
for i in range(len(sorted_years) - 1):
    current_year = sorted_years[i]
    next_year = sorted_years[i + 1]
    
    # Companies in current year but not in next year
    stopped = companies_by_year_dict[current_year] - companies_by_year_dict[next_year]
    
    if stopped:
        disappeared_companies[f"{current_year} -> {next_year}"] = sorted(list(stopped))

# Display companies that disappeared
for period, companies in disappeared_companies.items():
    print(f"\nCompanies that stopped reporting ({period}): {len(companies)}")
    print(companies[:10])  # Show first 10

# Create a summary dataframe of disappearing companies with their last year data
disappeared_summary = []
for company in {c for companies in disappeared_companies.values() for c in companies}:
    company_data = df_panel_data[df_panel_data['Company Name'] == company]
    last_year = company_data['Fiscal Year'].max()
    last_record = company_data[company_data['Fiscal Year'] == last_year].iloc[0] if len(company_data) > 0 else None
    if last_record is not None:
        disappeared_summary.append({
            'Company Name': company,
            'Last Report Year': int(last_year),
            'Revenue': last_record['Revenue'],
            'Operating Income': last_record['Operating Income (Loss)'],
            'Pretax Income': last_record['Pretax Income (Loss), Adj.'],
            'Net Cash from Operating': last_record['Net Cash from Operating Activities']
        })

disappeared_df = pd.DataFrame(disappeared_summary)
print("\n\nCompanies that stopped reporting - Last year metrics:")
print(disappeared_df.sort_values('Last Report Year', ascending=False).head(20))


Companies that stopped reporting (2008 -> 2009): 1
['Pennsylvania Real Estate Investment Trust']

Companies that stopped reporting (2010 -> 2011): 4
['COCA-COLA REFRESHMENTS USA, INC.', 'General Growth Properties Inc.', 'MF Global Holdings Ltd.', 'McAfee, Inc.']

Companies that stopped reporting (2011 -> 2012): 2
['CAMECO CORP', 'HEMACARE CORP /CA/']

Companies that stopped reporting (2012 -> 2013): 18
['AGNICO EAGLE MINES LTD', 'BARCLAY ROAD, INC.', 'BLUE COAT SYSTEMS INC', 'CYBRA CORP', 'Digital Domain Media Group, Inc.', 'Infusion Brands International, Inc.', 'JDA SOFTWARE GROUP INC', 'MICRONETICS INC', 'PEOPLES EDUCATIONAL HOLDINGS', 'RDA Holding Co.']

Companies that stopped reporting (2013 -> 2014): 22
['Actavis, Inc.', 'Aramark Services, Inc.', 'BLUE RIDGE REAL ESTATE CO', 'CelLynx Group, Inc.', 'DGT Holdings Corp.', 'EMERITUS CORP\\WA\\', 'EMTEC INC/NJ', 'Endo International', 'Forest Laboratories, LLC', 'GARDNER DENVER INC']

Companies that stopped reporting (2014 -> 2015): 19